# NB-04: Gestão de Banca e Juros Compostos

**Objetivo**: Determinar as melhores regras de gestão de banca, compound, stop gain/loss.

## Perguntas-chave:
1. Qual % de reinvestimento maximiza crescimento a longo prazo?
2. Qual stop-loss ótimo minimiza risco sem matar retorno?
3. Qual stop-gain ótimo equilibra frequência de saques e tamanho?
4. Qual o risk of ruin para cada configuração?
5. Como escalar a banca com segurança?

In [ ]:
import sys
sys.path.insert(0, '..')

# Carregar framework
%run 12_backtester.ipynb

In [ ]:
df = load_raw_data()
multipliers = df['multiplicador'].values
print(f"Dados: {len(multipliers):,} rounds")

## 1. Compound vs Flat: Impacto do Reinvestimento

In [ ]:
# Usar a melhor estratégia do NB-03 (ou martingale como baseline)
strategy_factory = lambda: MartingaleStrategy(trigger=6, target=2.0, pattern=[1, 2, 4])

# Testar: Flat vs Compound com diferentes %
compound_results = []

configs = [
    ('Flat (sem compound)', BankrollConfig(initial_bankroll=1000, base_bet_pct=0.0167,
                                           compound=False, stop_gain_pct=10.0)),
    ('Compound 25%', BankrollConfig(initial_bankroll=1000, base_bet_pct=0.0167,
                                    compound=True, compound_pct=0.25, stop_gain_pct=10.0)),
    ('Compound 50%', BankrollConfig(initial_bankroll=1000, base_bet_pct=0.0167,
                                    compound=True, compound_pct=0.50, stop_gain_pct=10.0)),
    ('Compound 75%', BankrollConfig(initial_bankroll=1000, base_bet_pct=0.0167,
                                    compound=True, compound_pct=0.75, stop_gain_pct=10.0)),
    ('Compound 100%', BankrollConfig(initial_bankroll=1000, base_bet_pct=0.0167,
                                     compound=True, compound_pct=1.0, stop_gain_pct=10.0)),
]

for name, cfg in configs:
    strat = strategy_factory()
    r = backtest(strat, multipliers, cfg)
    r.strategy_name = name
    compound_results.append(r)
    print(f"{name:25s} | Final: R${r.final_bankroll:>10,.2f} | "
          f"DD: {r.max_drawdown_pct:.1f}% | Apostas: {r.total_bets}")

plot_comparison(compound_results, 'Compound vs Flat')

## 2. Otimização do Stop Loss

In [ ]:
# Testar diferentes stop-loss
sl_values = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.80, 1.00]
sl_results = []

# Simular múltiplas sessões (blocos de 5000 rounds)
session_size = 5000
n_sessions = len(multipliers) // session_size

print(f"Simulando {n_sessions} sessões de {session_size} rounds cada")
print(f"{'SL%':>6} {'Sessões+':>10} {'Sessões-':>10} {'Win%':>8} {'Lucro Médio':>12} {'Lucro Total':>12}")
print("-" * 70)

sl_analysis = []

for sl in sl_values:
    session_profits = []

    for s in range(n_sessions):
        start = s * session_size
        end = start + session_size
        chunk = multipliers[start:end]

        cfg = BankrollConfig(
            initial_bankroll=1000, base_bet_pct=0.0167,
            compound=False, stop_loss_pct=sl, stop_gain_pct=0.20,
            session_rounds=session_size,
        )
        strat = strategy_factory()
        r = backtest(strat, chunk, cfg)
        session_profits.append(r.total_profit)

    profits = np.array(session_profits)
    win_sessions = (profits > 0).sum()
    loss_sessions = (profits <= 0).sum()
    win_pct = win_sessions / len(profits) * 100

    sl_analysis.append({
        'sl': sl, 'win_pct': win_pct, 'mean_profit': profits.mean(),
        'total_profit': profits.sum(), 'std': profits.std(),
        'median': np.median(profits),
    })

    print(f"{sl*100:>5.0f}% {win_sessions:>10} {loss_sessions:>10} "
          f"{win_pct:>7.1f}% R${profits.mean():>+10.2f} R${profits.sum():>+10.0f}")

sl_df = pd.DataFrame(sl_analysis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar([f"{int(sl*100)}%" for sl in sl_df['sl']], sl_df['win_pct'],
            color=COLORS['green'], alpha=0.8)
axes[0].set_title('% Sessões Lucrativas por Stop Loss')
axes[0].set_ylabel('% Sessões com Lucro')
axes[0].set_xlabel('Stop Loss')

axes[1].bar([f"{int(sl*100)}%" for sl in sl_df['sl']], sl_df['mean_profit'],
            color=[COLORS['green'] if p > 0 else COLORS['red'] for p in sl_df['mean_profit']],
            alpha=0.8)
axes[1].set_title('Lucro Médio por Sessão')
axes[1].set_ylabel('R$')
axes[1].set_xlabel('Stop Loss')

# Sharpe-like: mean/std
sharpe = sl_df['mean_profit'] / sl_df['std']
axes[2].bar([f"{int(sl*100)}%" for sl in sl_df['sl']], sharpe,
            color=COLORS['cyan'], alpha=0.8)
axes[2].set_title('Retorno/Risco (Sharpe) por Stop Loss')
axes[2].set_ylabel('Mean / Std')
axes[2].set_xlabel('Stop Loss')

plt.tight_layout()
plt.show()

## 3. Otimização do Stop Gain

In [ ]:
sg_values = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.00]

print(f"{'SG%':>6} {'Sessões que atingem':>20} {'Lucro Médio':>12} {'Rounds Médio':>15}")
print("-" * 60)

sg_analysis = []

for sg in sg_values:
    session_profits = []
    session_rounds_used = []

    for s in range(n_sessions):
        start = s * session_size
        end = start + session_size
        chunk = multipliers[start:end]

        cfg = BankrollConfig(
            initial_bankroll=1000, base_bet_pct=0.0167,
            compound=False, stop_loss_pct=0.50, stop_gain_pct=sg,
            session_rounds=session_size,
        )
        strat = strategy_factory()
        r = backtest(strat, chunk, cfg)
        session_profits.append(r.total_profit)
        session_rounds_used.append(len(r.equity_curve))

    profits = np.array(session_profits)
    rounds_arr = np.array(session_rounds_used)
    hit_target = (profits >= 1000 * sg * 0.9).sum()  # ~90% do target

    sg_analysis.append({
        'sg': sg, 'hit_pct': hit_target / len(profits) * 100,
        'mean_profit': profits.mean(), 'mean_rounds': rounds_arr.mean(),
        'total': profits.sum(),
    })

    print(f"{sg*100:>5.0f}% {hit_target:>10}/{n_sessions:<10} "
          f"R${profits.mean():>+10.2f} {rounds_arr.mean():>14.0f}")

sg_df = pd.DataFrame(sg_analysis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar([f"{int(sg*100)}%" for sg in sg_df['sg']], sg_df['hit_pct'],
            color=COLORS['green'], alpha=0.8)
axes[0].set_title('% Sessões que Atingem Stop Gain')
axes[0].set_ylabel('%')
axes[0].set_xlabel('Stop Gain')

# Lucro/round (eficiência)
efficiency = sg_df['mean_profit'] / sg_df['mean_rounds'] * 1000
axes[1].bar([f"{int(sg*100)}%" for sg in sg_df['sg']], efficiency,
            color=COLORS['yellow'], alpha=0.8)
axes[1].set_title('Eficiência: Lucro por 1000 Rounds')
axes[1].set_ylabel('R$ / 1000 rounds')
axes[1].set_xlabel('Stop Gain')

plt.tight_layout()
plt.show()

## 4. Risk of Ruin (Probabilidade de Perder Tudo)

In [ ]:
# Monte Carlo: simular 1000 sessões com dados shuffled
n_simulations = 1000
sim_session_size = 2000

configs_ror = [
    ('Conserv (1/2, SL50%)', [1, 2], 0.50),
    ('Moder (1/2/4, SL50%)', [1, 2, 4], 0.50),
    ('Agrss (1/2/4/8, SL50%)', [1, 2, 4, 8], 0.50),
    ('Moder (1/2/4, SL30%)', [1, 2, 4], 0.30),
    ('Moder (1/2/4, SL100%)', [1, 2, 4], 1.00),
]

print(f"Monte Carlo: {n_simulations} simulações de {sim_session_size} rounds")
print(f"{'Config':35s} {'Ruin%':>8} {'Sobrevive%':>12} {'Lucro Médio':>12} {'Mediana':>10}")
print("-" * 80)

for name, pattern, sl in configs_ror:
    ruin_count = 0
    final_bankrolls = []

    for sim in range(n_simulations):
        # Shuffle aleatório dos multiplicadores
        start = np.random.randint(0, len(multipliers) - sim_session_size)
        chunk = multipliers[start:start + sim_session_size]

        cfg = BankrollConfig(
            initial_bankroll=1000, base_bet_pct=0.0167,
            compound=False, stop_loss_pct=sl, stop_gain_pct=10.0,
            session_rounds=sim_session_size,
        )
        strat = MartingaleStrategy(trigger=6, target=2.0, pattern=pattern)
        r = backtest(strat, chunk, cfg)

        final_bankrolls.append(r.final_bankroll)
        if r.final_bankroll <= 0:
            ruin_count += 1

    fb = np.array(final_bankrolls)
    ruin_pct = ruin_count / n_simulations * 100
    survive_pct = 100 - ruin_pct
    mean_profit = fb.mean() - 1000
    median_profit = np.median(fb) - 1000

    print(f"{name:35s} {ruin_pct:>7.1f}% {survive_pct:>11.1f}% "
          f"R${mean_profit:>+10.2f} R${median_profit:>+8.2f}")

## 5. Estratégia de Scaling (Quando Aumentar a Banca)

In [ ]:
# Simular crescimento composto com saques parciais
# Regra: a cada stop-gain, sacar X% do lucro e reinvestir o resto

saque_pcts = [0.0, 0.25, 0.50, 0.75, 1.0]
n_cycles = 50  # 50 sessões de 5000 rounds

print(f"Simulação: {n_cycles} ciclos de {session_size} rounds")
print(f"Stop Gain: 20% | Stop Loss: 50%")
print(f"")

fig, ax = plt.subplots(figsize=(14, 6))

for saque_pct in saque_pcts:
    bankroll = 1000.0
    total_sacado = 0.0
    bankroll_history = [bankroll]

    for cycle in range(n_cycles):
        start = cycle * session_size % (len(multipliers) - session_size)
        chunk = multipliers[start:start + session_size]

        cfg = BankrollConfig(
            initial_bankroll=bankroll, base_bet_pct=0.0167,
            compound=False, stop_loss_pct=0.50, stop_gain_pct=0.20,
            session_rounds=session_size,
        )
        strat = strategy_factory()
        r = backtest(strat, chunk, cfg)

        new_bankroll = r.final_bankroll
        profit = new_bankroll - bankroll

        if profit > 0:
            saque = profit * saque_pct
            total_sacado += saque
            bankroll = new_bankroll - saque
        else:
            bankroll = new_bankroll

        if bankroll <= 0:
            bankroll_history.extend([0] * (n_cycles - cycle - 1))
            break

        bankroll_history.append(bankroll)

    label = f"Saque {saque_pct:.0%} (banca: R${bankroll:.0f}, sacado: R${total_sacado:.0f})"
    ax.plot(bankroll_history, linewidth=1.5, label=label)

ax.set_title(f'Crescimento da Banca com Diferentes % de Saque ({n_cycles} ciclos)')
ax.set_xlabel('Ciclo')
ax.set_ylabel('Banca (R$)')
ax.legend(fontsize=9)
ax.axhline(y=1000, color=COLORS['dim'], linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Tabela de Configuração Ótima por Perfil

In [ ]:
print("=" * 70)
print("CONFIGURAÇÃO RECOMENDADA POR PERFIL DE RISCO")
print("=" * 70)
print(f"")
print(f"Baseado em {len(multipliers):,} rounds históricos")
print(f"")

profiles = [
    {
        'name': 'CONSERVADOR',
        'pattern': '1/2',
        'trigger': 7,
        'base_bet': '1% da banca',
        'stop_gain': '10%',
        'stop_loss': '20%',
        'compound': 'Não',
        'saque': '100% do lucro',
        'desc': 'Baixo risco, ganhos pequenos e consistentes',
    },
    {
        'name': 'MODERADO',
        'pattern': '1/2/4',
        'trigger': 6,
        'base_bet': '1.67% da banca (banca/6)',
        'stop_gain': '20%',
        'stop_loss': '50%',
        'compound': '50% do lucro',
        'saque': '50% do lucro',
        'desc': 'Balanço entre risco e retorno',
    },
    {
        'name': 'AGRESSIVO',
        'pattern': '1/2/4/8',
        'trigger': 6,
        'base_bet': '1.67% da banca',
        'stop_gain': '30%',
        'stop_loss': '80%',
        'compound': '75% do lucro',
        'saque': '25% do lucro',
        'desc': 'Alto risco, potencial de crescimento rápido',
    },
]

for p in profiles:
    print(f"--- {p['name']} ---")
    print(f"  Setup:     {p['pattern']}")
    print(f"  Trigger:   {p['trigger']} LOWs")
    print(f"  Aposta:    {p['base_bet']}")
    print(f"  Stop Gain: {p['stop_gain']}")
    print(f"  Stop Loss: {p['stop_loss']}")
    print(f"  Compound:  {p['compound']}")
    print(f"  Saque:     {p['saque']}")
    print(f"  → {p['desc']}")
    print()